# 🎙️ Voice Product Discovery — Executable System Walkthrough

A voice-to-voice product discovery agent: **Whisper** turns speech into text, a **LangGraph** multi-agent pipeline (router → planner → retriever → answerer/critic) decides which **MCP tools** to call — `rag.search` over a private catalog built from the **Amazon Product Dataset 2020** and `web.search` for live prices — reconciles catalog-vs-web conflicts, and replies with a ~15-second **spoken** summary plus citations and a comparison table in a React web app.

This notebook is the system, inside-out. Each section pairs a capability with the **actual source file that implements it** (rendered straight from the repository — the very code that executes) and, where it makes sense, a **live demonstration cell** whose output you can read below it. The final cells build the web UI and publish the running app at a public HTTPS URL.

**To run it:**
1. *(Recommended)* Sidebar **🔑 Secrets** → add `OPENAI_API_KEY` → enable **Notebook access**. The key stays in your Google account only. Without it, the pipeline runs on a clearly-labeled deterministic mock so everything still executes.
2. **Runtime → Run all.** First run takes ~8–12 min (dataset download, embedding build, Whisper model).
3. Open the `https://….trycloudflare.com` URL printed at the end and speak to the app.

## System Architecture

```
Browser (React) ── /api/transcribe ─▶ Whisper ASR (faster-whisper | OpenAI)
      │              /api/discover ─▶ LangGraph:
      │                               router → [safety] → planner
      │                                 → retrieve(rag.search + rerank)
      │                                 → [web_compare → reconcile | web_fallback]
      │                                 → answerer/critic
      │              /api/speak ─────▶ TTS (edge-tts | OpenAI) → mp3
      │                                       │
      └── step log · table · citations   MCP client ── stdio JSON-RPC ──▶ MCP server
                                                       web.search (cache + rate limit + allowlist)
                                                       rag.search (Chroma hybrid retrieval)
```

```
backend/app          FastAPI gateway (/api/transcribe · /api/discover · /api/speak · /api/health)
backend/graph        LangGraph state, nodes, wiring, model-agnostic LLM layer
backend/mcp_server   MCP server (web.search, rag.search) + stdio client
backend/rag          CSV → parquet + Chroma ingestion, embedders, hybrid retrieval
backend/speech       Whisper ASR + TTS
prompts/             every runtime prompt (loaded by the nodes at run time)
frontend/            React app: mic, transcript, agent step log, table, citations, audio
```

## Environment Setup

Clone the repository, install Node + Python dependencies, and choose providers. With an `OPENAI_API_KEY` secret the language model is `gpt-4o-mini`; otherwise a deterministic mock keeps the walkthrough fully executable.

In [15]:
REPO_URL = "https://github.com/aimanaltoubi/voice-product-discovery.git"  # @param {type:"string"}

import pathlib, re, subprocess
name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
clone_url = REPO_URL
try:  # optional: GITHUB_TOKEN secret for private repos
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        clone_url = re.sub(r"^https://", f"https://{tok}@", REPO_URL)
except Exception:
    pass
if not pathlib.Path(name).exists():
    subprocess.run(["git", "clone", "--depth", "1", clone_url, name], check=True)
%cd {name}
REPO = pathlib.Path.cwd()
print("Repo ready at", REPO)

/content/voice-product-discovery/voice-product-discovery
Repo ready at /content/voice-product-discovery/voice-product-discovery


In [16]:
%%bash
# Node 18+ for the Vite build, plus all backend Python dependencies.
set -e
if ! node -e 'process.exit(parseInt(process.versions.node)>=18?0:1)' 2>/dev/null; then
  echo "Installing Node 20…"
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
  apt-get install -y nodejs > /dev/null 2>&1
fi
echo "node $(node --version)"
echo "Installing backend dependencies (a few minutes)…"
pip install -q -r backend/requirements.txt kagglehub
echo "Backend deps installed."

node v20.19.0
Installing backend dependencies (a few minutes)…
Backend deps installed.


In [17]:
# Provider configuration — reads OPENAI_API_KEY from Colab Secrets, mock fallback.
import os
key = None
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception:
    key = None
if key:
    os.environ["OPENAI_API_KEY"] = key
    os.environ["LLM_PROVIDER"] = "openai"
    os.environ.setdefault("LLM_MODEL", "gpt-4o-mini")
    print("✅ OPENAI_API_KEY found in Colab Secrets → real LLM mode (" + os.environ["LLM_MODEL"] + ").")
else:
    os.environ["LLM_PROVIDER"] = "mock"
    print("⚠️ No OPENAI_API_KEY secret → keyless MOCK mode (deterministic demo heuristics).")
    print("   Add the secret via the 🔑 sidebar, enable Notebook access, and re-run this cell.")
os.environ.setdefault("EMBEDDINGS_PROVIDER", "local")  # keyless ONNX MiniLM
os.environ.setdefault("ASR_PROVIDER", "local")         # faster-whisper on CPU
os.environ.setdefault("TTS_PROVIDER", "edge")          # keyless Edge voices
print("embeddings=local · asr=local(faster-whisper) · tts=edge")

✅ OPENAI_API_KEY found in Colab Secrets → real LLM mode (gpt-4o-mini).
embeddings=local · asr=local(faster-whisper) · tts=edge


In [18]:
# Walkthrough helpers: render repo source files inline, pretty-print agent runs.
import json, sys
from pathlib import Path
from IPython.display import Markdown, display

sys.path.insert(0, str(REPO / "backend"))

LANG = {".py": "python", ".md": "markdown", ".js": "javascript", ".jsx": "jsx", ".sh": "bash"}

def show(rel_path):
    """Render a source file exactly as it exists in the repository —
    the same file the system imports and executes."""
    p = REPO / rel_path
    fence = "`" * 6
    display(Markdown(f"### 📄 `{rel_path}`\n{fence}{LANG.get(p.suffix, '')}\n{p.read_text()}\n{fence}"))

def js(x, n=900):
    s = json.dumps(x, indent=1, ensure_ascii=False, default=str)
    return s if len(s) <= n else s[:n] + " …"

def print_steps(p):
    print(f'🎤 "{p["transcript"]}"\n')
    for s in p["steps"]:
        print(f'── {s["name"]}  [{s["timestamp"]}]')
        print("   out:", js(s["output"], 650).replace("\n", "\n        "), "\n")
    print("🔊 spoken_answer:", p["spoken_answer"])
    if p.get("top_pick"):
        print("⭐ top_pick:", p["top_pick"]["title"], "—", p["top_pick"].get("price"))
    if p["comparison_table"]:
        print("📊 comparison_table:")
        for r in p["comparison_table"]:
            print(f'   • {r["doc_id"]}: {str(r["title"])[:58]} | price {r.get("price")} | ⭐ {r.get("rating")} | $/oz {r.get("price_per_oz")}')
    print("🔗 citations:", [(c.get("doc_id") or c.get("url")) for c in p["citations"]])
    print("⛔ blocked:", p["blocked"], "| source:", p["source"])

print("Walkthrough helpers ready.")

Walkthrough helpers ready.


## Model-Agnostic Language-Model Layer

One environment variable swaps the LLM (OpenAI, Anthropic, Google, local Ollama) with no code changes — every reasoning node calls the model through LangChain's `init_chat_model` with schema-validated structured output. The same file contains the deterministic `mock` provider used for keyless runs, so the difference is always visible and labeled.

In [19]:
show("backend/graph/llm.py")

### 📄 `backend/graph/llm.py`
``````python
"""Model-agnostic LLM access.

The graph nodes never talk to a provider SDK directly; they call
`call_structured(prompt, Schema, node=...)`. The provider/model is selected via
env (LLM_PROVIDER / LLM_MODEL) and instantiated through LangChain's
`init_chat_model`, so swapping OpenAI <-> Anthropic <-> Gemini <-> Ollama is a
config change only (brief: "Model-Agnostic Requirement").

`LLM_PROVIDER=mock` is a deterministic, keyless heuristic used for offline
demos / CI. It is not a language model and every log entry marks it as such.
"""
from __future__ import annotations

import re
from functools import lru_cache
from typing import Any, Type, TypeVar

from pydantic import BaseModel

from app.config import settings

T = TypeVar("T", bound=BaseModel)

_PROVIDER_MAP = {
    "openai": "openai",
    "anthropic": "anthropic",
    "google": "google_genai",
    "google_genai": "google_genai",
    "gemini": "google_genai",
    "ollama": "ollama",
}


@lru_cache(maxsize=1)
def get_chat_model():
    """Instantiate the chat model once, from env config."""
    from langchain.chat_models import init_chat_model

    provider = _PROVIDER_MAP.get(settings.LLM_PROVIDER)
    if provider is None:
        raise ValueError(
            f"Unknown LLM_PROVIDER={settings.LLM_PROVIDER!r}. "
            "Use openai | anthropic | google_genai | ollama | mock."
        )
    return init_chat_model(settings.LLM_MODEL, model_provider=provider, temperature=0)


async def call_structured(
    prompt: str,
    schema: Type[T],
    *,
    node: str,
    context: dict[str, Any] | None = None,
) -> T:
    """Run one structured LLM call and return a validated `schema` instance.

    `context` carries the raw state a node already has; it is only consumed by
    the mock provider (real providers read the rendered prompt).
    """
    if settings.LLM_PROVIDER == "mock":
        return _mock_structured(schema, context or {})
    model = get_chat_model().with_structured_output(schema)
    return await model.ainvoke(prompt)


# --------------------------------------------------------------------------
# Mock provider: deterministic heuristics per output schema. Keyless demo/CI.
# --------------------------------------------------------------------------

_WORD_NUMBERS = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
    "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
    "thirteen": 13, "fourteen": 14, "fifteen": 15, "sixteen": 16,
    "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
    "twenty-five": 25, "thirty": 30, "forty": 40, "fifty": 50, "hundred": 100,
}
_MATERIALS = [
    "stainless steel", "stainless", "glass", "granite", "marble", "wood",
    "tile", "ceramic", "leather", "carpet", "fabric",
]
_SAFETY_PATTERNS = [
    (r"mix(ing)?\b.*(bleach|ammonia)", "unsafe_chemical_mixing"),
    (r"(bleach|ammonia)\b.*mix", "unsafe_chemical_mixing"),
    (r"\b(drink|ingest|swallow|eat)\b", "ingestion_risk"),
    (r"\b(explosive|weapon|poison(ing)? (someone|a person))\b", "harmful_intent"),
]
_LIVE_PATTERNS = r"\b(current|latest|right now|today|in stock|availability|available now|live price)\b"


def _parse_budget(text: str) -> float | None:
    m = re.search(r"(?:under|below|less than|max(?:imum)?|up to)\s*\$?\s*(\d+(?:\.\d{1,2})?)", text)
    if m:
        return float(m.group(1))
    m = re.search(r"\$\s?(\d+(?:\.\d{1,2})?)", text)
    if m:
        return float(m.group(1))
    m = re.search(
        r"(?:under|below|less than|up to)\s+([a-z\-]+)\s+(?:dollars|bucks)", text
    )
    if m and m.group(1) in _WORD_NUMBERS:
        return float(_WORD_NUMBERS[m.group(1)])
    return None


def _mock_structured(schema: Type[T], ctx: dict[str, Any]) -> T:
    name = schema.__name__
    if name == "RouterOutput":
        t = (ctx.get("transcript") or "").lower()
        material = next((m for m in _MATERIALS if m in t), None)
        flags = sorted({f for pat, f in _SAFETY_PATTERNS if re.search(pat, t)})
        return schema(
            task="product_recommendation",
            constraints={
                "budget": _parse_budget(t),
                "material": material,
                "brand": None,
                "category": "cleaner" if "clean" in t else None,
                "eco_friendly": bool(re.search(r"\b(eco|green|natural|plant[- ]based|non[- ]toxic)\b", t)) or None,
            },
            safety_flags=flags,
            needs_live=bool(re.search(_LIVE_PATTERNS, t)),
        )
    if name == "PlanOutput":
        router = ctx.get("router") or {}
        c = router.get("constraints") or {}
        sources = ["rag.search"] + (["web.search"] if router.get("needs_live") else [])
        return schema(
            sources=sources,
            retrieval_filters={
                "category": c.get("category"),
                "max_price": c.get("budget"),
                "material": c.get("material"),
                "eco_friendly": c.get("eco_friendly"),
            },
            comparison_criteria=["price", "rating", "ingredients", "eco-friendliness"],
        )
    if name == "RerankOutput":
        cands = ctx.get("candidates") or []
        ranked = sorted(
            cands,
            key=lambda p: (-(p.get("rating") or 0), p.get("price") or 1e9),
        )
        return schema(
            ranked_doc_ids=[p["doc_id"] for p in ranked[:3]],
            rationale="[mock] Ranked by rating (desc) then price (asc) within the applied filters.",
        )
    if name == "AnswerOutput":
        picks = ctx.get("top_picks") or []
        if not picks:
            return schema(
                spoken_answer="I couldn't find a matching product in the catalog or on the web. Try rephrasing your request.",
                top_pick_doc_id="",
                citation_doc_ids=[],
            )
        top = picks[0]
        price = f"${top['price']:.2f}" if isinstance(top.get("price"), (int, float)) else "an unlisted price"
        rating = f"{top['rating']:.1f} stars" if isinstance(top.get("rating"), (int, float)) else "no rating yet"
        return schema(
            spoken_answer=(
                f"My top pick is {top.get('title')} by {top.get('brand') or 'an unbranded maker'} — "
                f"{rating}, typically {price}. I compared {len(picks)} options against your criteria; "
                "details and sources are on your screen. Would you like the most affordable or the highest rated?"
            ),
            top_pick_doc_id=top["doc_id"],
            citation_doc_ids=[p["doc_id"] for p in picks],
        )
    raise ValueError(f"Mock provider has no heuristic for schema {name}")

``````

## Prompt Design — Every Runtime Prompt, Mapped to Its Node

The `prompts/` folder is not documentation of the prompts — it **is** the prompts: `graph/prompts.py` loads these files at run time and fills the `<<placeholders>>`. Below: the shared system prompt, the router with its few-shot examples, the planner (including its tool-selection policy), the reranker, the answerer/critic, and the mapping of each prompt to its node and output schema.

In [20]:
for f in ["prompts/system.md", "prompts/router.md", "prompts/few_shots_router.md",
          "prompts/planner.md", "prompts/reranker.md", "prompts/answerer.md",
          "prompts/README.md"]:
    show(f)

### 📄 `prompts/system.md`
``````markdown
You are one agent inside a voice-to-voice product-discovery assistant for e-commerce.
Global rules that apply to every agent in the pipeline:
- Ground every claim ONLY in the data provided in the prompt (private catalog rows identified by doc_id, or live web results identified by URL). Never invent products, prices, ratings, or ingredients.
- If evidence is missing, say so instead of guessing.
- Never give unsafe chemical advice (mixing chemicals, ingestion, misuse). Household-cleaning safety questions must be redirected to product labels and poison control.
- Answers must be concise: the final spoken summary is read aloud by TTS and must fit in about 15 seconds (~40 words).
- Return outputs strictly in the requested JSON schema.

``````

### 📄 `prompts/router.md`
``````markdown
ROLE: Router (Intent Classifier) — first node of the LangGraph pipeline.

From the user's spoken request, extract:
- task: a short label for what the user wants (e.g. "product_recommendation", "price_check").
- constraints: budget in USD if stated (convert number words like "fifteen dollars" to 15), material (e.g. "stainless steel"), brand, product category, eco_friendly preference (true only if explicitly implied). Use null for anything not stated.
- safety_flags: list of flags if the request involves unsafe chemical advice (e.g. mixing bleach and ammonia), ingestion of cleaning products, or other harmful intent. Empty list if safe.
- needs_live: true ONLY if the user explicitly asks for current/latest price, availability, "right now", "in stock", or similar live information.

<<few_shots>>

User spoken request: "<<transcript>>"

``````

### 📄 `prompts/few_shots_router.md`
``````markdown
Examples:

Request: "Recommend an eco-friendly stainless-steel cleaner under fifteen dollars."
Output: {"task": "product_recommendation", "constraints": {"budget": 15, "material": "stainless steel", "brand": null, "category": "cleaner", "eco_friendly": true}, "safety_flags": [], "needs_live": false}

Request: "What's the current price of Weiman glass cleaner, is it in stock?"
Output: {"task": "price_check", "constraints": {"budget": null, "material": "glass", "brand": "Weiman", "category": "cleaner", "eco_friendly": null}, "safety_flags": [], "needs_live": true}

Request: "Can I mix bleach and ammonia to make a stronger cleaner?"
Output: {"task": "unsafe_chemistry_question", "constraints": {"budget": null, "material": null, "brand": null, "category": null, "eco_friendly": null}, "safety_flags": ["unsafe_chemical_mixing"], "needs_live": false}

``````

### 📄 `prompts/planner.md`
``````markdown
ROLE: Planner — second node of the LangGraph pipeline.

Decide, based on the Router's output:
- sources: which MCP tools to call. ALWAYS include "rag.search" (private catalog is the primary source of facts). Include "web.search" ONLY if needs_live is true (the user asked for current price / availability / latest info). This is the planner rubric from the project brief: "prefer rag.search for facts; if user asks current price/availability/now/latest, also call web.search."
- retrieval_filters: metadata filters for the private catalog — category (a short catalog-style category term), max_price (from budget), material, eco_friendly. Use null for filters that don't apply.
- comparison_criteria: 3-5 criteria to weigh the candidates against, derived from the user's constraints (e.g. price, rating, ingredients, eco-friendliness, value per ounce).

Router output (JSON):
<<router_json>>

Original user request: "<<transcript>>"

``````

### 📄 `prompts/reranker.md`
``````markdown
ROLE: Retriever reranker — runs inside the rag.search step of the LangGraph pipeline, AFTER hybrid retrieval (vector similarity + metadata filters) has produced the candidate set below.

Rank the candidates by true relevance to the user's request, considering the comparison criteria. Return:
- ranked_doc_ids: the doc_id values of the TOP 3 candidates, best first. Only use doc_id values that appear in the candidate list.
- rationale: one or two sentences explaining the ranking.

User request: "<<transcript>>"
Comparison criteria: <<criteria_json>>
Candidates (JSON, from hybrid retrieval):
<<candidates_json>>

``````

### 📄 `prompts/answerer.md`
``````markdown
ROLE: Answerer/Critic — final node of the LangGraph pipeline.

Synthesize ONE concise spoken recommendation (about 40 words, ~15 seconds when read aloud) for the user's request. Requirements:
- Ground ONLY in the provided product rows (and the live web result, if present). Do not add facts that are not in the data.
- Name the top pick with brand, price, rating, and one key ingredient/feature detail.
- Mention that you compared alternatives ("I compared this with N alternatives").
- Mention that details and sources were sent to the screen.
- End with a short follow-up question offering "the most affordable or the highest rated".
- CRITIC DUTY: if the reconciliation data flags a discrepancy between catalog and live data (price difference, availability), you MUST explicitly mention it in the spoken answer (e.g. "note: the live price differs from our catalog").
- Never include unsafe chemical advice.

Return:
- spoken_answer: the ~40-word spoken summary.
- top_pick_doc_id: doc_id of your top pick (must be one of the provided rows).
- citation_doc_ids: every doc_id you relied on.

User request: "<<transcript>>"
Comparison criteria: <<criteria_json>>
Mode: <<mode>>  (private = catalog rows; web_fallback = live web rows because the private catalog had no match)
Product rows (JSON):
<<products_json>>
Live web comparison + reconciliation (JSON, may be null):
<<web_json>>

``````

### 📄 `prompts/README.md`
``````markdown
# Prompt Disclosure

This folder satisfies the brief's **Prompt Disclosure** requirement (5 pts):
it contains every prompt used by the agents, and these files are **not copies**
— they are loaded at runtime by `backend/graph/prompts.py`, so what you read
here is exactly what the LLM receives.

Placeholders use `<<name>>` syntax and are substituted by the node before the
LLM call (see `render()` in `backend/graph/prompts.py`).

## Prompt → node / tool mapping

| Prompt file | LangGraph node | Structured output schema | Placeholders |
|---|---|---|---|
| `system.md` | prepended to **every** LLM call (system message) | — | — |
| `router.md` | `router` (Intent Classifier) | `RouterOutput` | `<<transcript>>`, `<<few_shots>>` |
| `few_shots_router.md` | injected into `router.md` | — | — |
| `planner.md` | `planner` | `PlanOutput` | `<<router_json>>`, `<<transcript>>` |
| `reranker.md` | `retrieve` (the LLM rerank stage of `rag.search`) | `RerankOutput` | `<<transcript>>`, `<<criteria_json>>`, `<<candidates_json>>` |
| `answerer.md` | `answer` (Answerer/Critic) | `AnswerOutput` | `<<transcript>>`, `<<criteria_json>>`, `<<mode>>`, `<<products_json>>`, `<<web_json>>` |

## Non-LLM (deterministic) steps

Three pipeline steps intentionally use **no prompt** because they are
deterministic code, which is more auditable than an LLM for these jobs:

- `safety` — hard block + fixed safe refusal whenever the router raised
  `safety_flags` (`backend/graph/nodes.py`).
- `reconcile` — SKU/brand/title fuzzy matching between catalog and live web
  results + price-delta discrepancy flags (`backend/graph/nodes.py`).
- MCP tools `web.search` and `rag.search` — pure retrieval, no generation
  (`backend/mcp_server/server.py`).

## Planner rubric (as required by the brief)

The planner rule from the brief — *"prefer rag.search for facts; if user asks
'current price/availability/now/latest,' also call web.search"* — is stated in
`planner.md` **and** enforced deterministically after the LLM call in
`planner_node` (the graph guarantees `rag.search` is always present and adds
`web.search` whenever the router set `needs_live=true`, even if the LLM plan
omitted it). Enforcement events appear in the agent step log.

``````

## Private Catalog — Amazon Product Dataset 2020, Embeddings, and Value Normalization

The ingestion pipeline reads the Kaggle **Amazon Product Dataset 2020** CSV, normalizes it into `products.parquet` / `reviews.parquet`, derives `price_per_oz` (parsing sizes like *"2 x 16 oz"*), flags eco-friendly items with a negation-aware heuristic, and embeds `title + features + review snippets` into a Chroma index.

The cell after the source downloads the **real dataset directly from Kaggle at run time** (public dataset — no account needed) and builds the index from it. The CSV lives only on this VM under `data/raw/`; the repository never redistributes it. If Kaggle is unreachable, the bundled synthetic sample keeps the walkthrough running.

In [21]:
show("backend/rag/ingest.py")
show("backend/rag/embeddings.py")

### 📄 `backend/rag/ingest.py`
``````python
"""Data preprocessing + indexing for the private catalog (brief: Data Section).

Pipeline:  CSV  ->  products.parquet (+ reviews.parquet)  ->  Chroma index

Sources:
  --sample            data/sample_products.csv (synthetic seed catalog, ships
                      with the repo so the app runs out of the box)
  --csv PATH          the real Kaggle "Amazon Product Dataset 2020" CSV
                      (marketing_sample_for_amazon_com-ecommerce_*.csv).
                      Column names are auto-mapped (see COLUMN_CANDIDATES).

Per row we compute:
  - a stable doc_id (kept if present, else AMZ2020-<sha1[:10]> of the source id)
  - size_oz + price_per_oz  (unit normalization: "Normalize units (e.g., price
    per oz) to support fair comparisons")
  - eco_friendly flag (keyword heuristic over title+features+ingredients)
  - the embedding text = title + features + ingredients + review snippets

Run from backend/:
  python -m rag.ingest --sample
  python -m rag.ingest --csv ../data/raw/amazon2020.csv --category "Household" --limit 3000
"""
from __future__ import annotations

import argparse
import hashlib
import json
import re
import sys
from pathlib import Path

import pandas as pd

from app.config import CHROMA_DIR, DATA_DIR, PROCESSED_DIR, settings
from rag.embeddings import get_embedder

COLUMN_CANDIDATES: dict[str, list[str]] = {
    "id": ["doc_id", "Uniq Id", "uniq_id", "id", "asin"],
    "title": ["title", "Product Name", "name", "product_name"],
    "brand": ["brand", "Brand Name", "Brand", "manufacturer"],
    "category": ["category", "Category", "Amazon Category and Sub-category"],
    "price": ["price", "Selling Price", "List Price", "selling_price"],
    "rating": ["rating", "Average Rating", "stars", "average_review_rating"],
    "features": ["features", "About Product", "about_product", "Product Description", "description"],
    "ingredients": ["ingredients", "Product Specification", "product_specification"],
    "reviews": ["review_snippets", "reviews", "Customer Reviews", "customer_reviews"],
}

ECO_KEYWORDS = (
    "eco", "plant-based", "plant based", "biodegradable", "non-toxic",
    "nontoxic", "natural", "green seal", "epa safer choice", "vegan",
    "phosphate-free", "sustainab",
)

_ECO_NEG = re.compile(
    r"\b(?:not|isn'?t|no|never|without)\s+(?:\w+[- ]){0,2}?"
    r"(?:eco|plant[- ]based|biodegradable|non[- ]?toxic|natural|vegan|sustainab)",
    re.I,
)


def is_eco_friendly(blob: str) -> bool:
    """Keyword heuristic with basic negation handling.

    'plant-based formula' -> True; 'not plant-based' -> that mention is
    negated and doesn't count. A product is flagged only if at least one
    non-negated eco keyword remains.
    """
    cleaned = _ECO_NEG.sub(" ", blob)
    return any(k in cleaned for k in ECO_KEYWORDS)

_OZ_PER = {"oz": 1.0, "fl oz": 1.0, "floz": 1.0, "ounce": 1.0,
           "ml": 1.0 / 29.5735, "l": 33.814, "liter": 33.814, "litre": 33.814,
           "gal": 128.0, "gallon": 128.0, "qt": 32.0, "quart": 32.0,
           "pt": 16.0, "pint": 16.0, "lb": 16.0, "pound": 16.0}

_SIZE_RE = re.compile(
    r"(?:(\d+)\s*(?:x|pack of|pk of)\s*)?"
    r"(\d+(?:\.\d+)?)\s*(fl\.?\s*oz|floz|oz|ounce|ml|liter|litre|l|gallon|gal|quart|qt|pint|pt|pound|lb)s?\b",
    re.IGNORECASE,
)


def parse_size_oz(text: str) -> float | None:
    """Extract total fluid-ounce size from free text (handles '2 x 16 oz')."""
    if not text:
        return None
    best = None
    for m in _SIZE_RE.finditer(text):
        mult = float(m.group(1)) if m.group(1) else 1.0
        qty = float(m.group(2))
        unit = re.sub(r"[.\s]", "", m.group(3).lower())
        unit = {"floz": "oz", "ounce": "oz", "liter": "l", "litre": "l",
                "gallon": "gal", "quart": "qt", "pint": "pt", "pound": "lb"}.get(unit, unit)
        per = _OZ_PER.get(unit)
        if per:
            oz = round(mult * qty * per, 2)
            best = max(best, oz) if best else oz
    return best


def _first_price(value) -> float | None:
    if value is None:
        return None
    m = re.search(r"(\d{1,5}(?:\.\d{1,2})?)", str(value).replace(",", ""))
    return float(m.group(1)) if m else None


def _pick(df: pd.DataFrame, field: str) -> str | None:
    lower = {c.lower(): c for c in df.columns}
    for cand in COLUMN_CANDIDATES[field]:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def load_and_normalize(csv_path: Path, category_filter: str | None, limit: int | None) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(csv_path, dtype=str, on_bad_lines="skip", engine="python")
    cols = {f: _pick(df, f) for f in COLUMN_CANDIDATES}
    if not cols["title"]:
        raise SystemExit(f"Could not find a title column in {csv_path.name}. Columns: {list(df.columns)[:15]}")

    def val(row, field):
        c = cols[field]
        v = row.get(c) if c else None
        return None if v is None or (isinstance(v, float) and pd.isna(v)) or str(v).lower() == "nan" else str(v).strip()

    products, reviews = [], []
    for _, row in df.iterrows():
        title = val(row, "title")
        if not title:
            continue
        category = val(row, "category") or "Uncategorized"
        if category_filter and category_filter.lower() not in (category + " " + title).lower():
            continue
        raw_id = val(row, "id") or title
        doc_id = raw_id if raw_id.startswith(("SAMPLE-", "AMZ2020-")) else \
            "AMZ2020-" + hashlib.sha1(raw_id.encode()).hexdigest()[:10]
        price = _first_price(val(row, "price"))
        rating_raw = val(row, "rating")
        rating = None
        if rating_raw:
            m = re.search(r"(\d(?:\.\d)?)", rating_raw)
            rating = min(float(m.group(1)), 5.0) if m else None
        features = (val(row, "features") or "")[:1200]
        ingredients = (val(row, "ingredients") or "")[:800]
        snippets = (val(row, "reviews") or "")[:800]
        blob = f"{title} {features} {ingredients}".lower()
        size_oz = parse_size_oz(f"{title} {features}")
        products.append({
            "id": doc_id, "doc_id": doc_id, "title": title,
            "brand": val(row, "brand") or "Unknown", "category": category,
            "price": price, "rating": rating, "features": features,
            "ingredients": ingredients,
            "eco_friendly": is_eco_friendly(blob),
            "size_oz": size_oz,
            "price_per_oz": round(price / size_oz, 4) if price and size_oz else None,
            "review_snippets": snippets,
        })
        for sn in [s.strip() for s in snippets.split("|") if s.strip()]:
            reviews.append({"product_id": doc_id, "stars": rating, "summary": sn})
        if limit and len(products) >= limit:
            break
    return pd.DataFrame(products), pd.DataFrame(reviews)


def build_index(products: pd.DataFrame) -> None:
    import chromadb

    embedder = get_embedder()
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    try:
        client.delete_collection(settings.CHROMA_COLLECTION)
    except Exception:
        pass
    col = client.create_collection(settings.CHROMA_COLLECTION, metadata={"embedder": embedder.name})

    ids, docs, metas = [], [], []
    for _, p in products.iterrows():
        ids.append(p["doc_id"])
        # Embedding text per brief: title + features + selected review snippets
        # (+ ingredients). Stored lowercased so where_document $contains
        # matching is case-insensitive; display fields live in metadata.
        docs.append(
            f"{p['title']} | {p['brand']} | {p['category']} | "
            f"{p['features']} | {p['ingredients']} | {p['review_snippets']}".lower()[:4000]
        )
        meta = {
            "doc_id": p["doc_id"], "title": p["title"], "brand": p["brand"],
            "category": p["category"], "eco_friendly": bool(p["eco_friendly"]),
            "features": (p["features"] or "")[:400],
            "ingredients": (p["ingredients"] or "")[:300],
        }
        for numf in ("price", "rating", "size_oz", "price_per_oz"):
            v = p[numf]
            if v is not None and not pd.isna(v):
                meta[numf] = float(v)
        metas.append(meta)

    B = 64
    for i in range(0, len(ids), B):
        embs = embedder.encode(docs[i : i + B])
        col.add(ids=ids[i : i + B], documents=docs[i : i + B],
                metadatas=metas[i : i + B], embeddings=embs)
        print(f"  indexed {min(i + B, len(ids))}/{len(ids)}", file=sys.stderr)

    (CHROMA_DIR.parent / "catalog_meta.json").write_text(json.dumps({
        "embedder": embedder.name,
        "count": len(ids),
        "categories": sorted(products["category"].dropna().unique().tolist()),
    }, indent=2))


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Build the private-catalog parquet files + Chroma index.")
    src = ap.add_mutually_exclusive_group(required=True)
    src.add_argument("--sample", action="store_true", help="use the bundled synthetic sample catalog")
    src.add_argument("--csv", type=Path, help="path to the Kaggle Amazon 2020 CSV")
    ap.add_argument("--category", default=None, help="substring filter, e.g. 'Household' (curated slice)")
    ap.add_argument("--limit", type=int, default=None, help="max products to index")
    args = ap.parse_args(argv)

    csv_path = DATA_DIR / "sample_products.csv" if args.sample else args.csv
    if not csv_path.exists():
        raise SystemExit(f"CSV not found: {csv_path}")

    print(f"Loading {csv_path} ...", file=sys.stderr)
    products, reviews = load_and_normalize(csv_path, args.category, args.limit)
    if products.empty:
        raise SystemExit("No products matched — check --category / the CSV columns.")

    products.to_parquet(PROCESSED_DIR / "products.parquet", index=False)
    if not reviews.empty:
        reviews.to_parquet(PROCESSED_DIR / "reviews.parquet", index=False)
    print(f"Wrote {len(products)} products -> {PROCESSED_DIR/'products.parquet'}", file=sys.stderr)

    print(f"Building Chroma index with embedder '{settings.EMBEDDINGS_PROVIDER}' ...", file=sys.stderr)
    build_index(products)
    print("Done. Index at", CHROMA_DIR, file=sys.stderr)


if __name__ == "__main__":
    main()

``````

### 📄 `backend/rag/embeddings.py`
``````python
"""Embedding providers for the private-catalog index.

- local  : sentence-transformers/all-MiniLM-L6-v2 exported to ONNX (Chroma's
           bundled embedding function). Downloads once (~80 MB), then offline.
- openai : text-embedding-3-small via the OpenAI API.
- hash   : deterministic pseudo-embeddings for CI/offline smoke tests ONLY —
           they preserve token overlap, not semantics. Never use for the demo.

The ingest step records which embedder built the index; retrieval refuses to
query with a different one (a silent mismatch would corrupt similarity).
"""
from __future__ import annotations

import hashlib
import math
import re

from app.config import settings


class Embedder:
    name: str = "base"

    def encode(self, texts: list[str]) -> list[list[float]]:  # pragma: no cover
        raise NotImplementedError


class LocalMiniLMEmbedder(Embedder):
    name = "local-minilm-l6-v2"

    def __init__(self) -> None:
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

        self._ef = ONNXMiniLM_L6_V2()

    def encode(self, texts: list[str]) -> list[list[float]]:
        try:
            out = self._ef(input=texts)
        except TypeError:  # older chroma EF signature
            out = self._ef(texts)
        return [list(map(float, v)) for v in out]


class OpenAIEmbedder(Embedder):
    def __init__(self) -> None:
        from openai import OpenAI

        self._client = OpenAI()
        self._model = settings.OPENAI_EMBEDDING_MODEL
        self.name = f"openai-{self._model}"

    def encode(self, texts: list[str]) -> list[list[float]]:
        out: list[list[float]] = []
        for i in range(0, len(texts), 100):
            batch = [t[:6000] for t in texts[i : i + 100]]
            resp = self._client.embeddings.create(model=self._model, input=batch)
            out.extend([d.embedding for d in resp.data])
        return out


class HashEmbedder(Embedder):
    """Token-hash bag-of-words vectors. TEST/CI ONLY (documented)."""

    name = "hash-384-test-only"
    DIM = 384

    def encode(self, texts: list[str]) -> list[list[float]]:
        vecs = []
        for text in texts:
            v = [0.0] * self.DIM
            for tok in re.findall(r"[a-z0-9]+", text.lower()):
                h = int(hashlib.sha256(tok.encode()).hexdigest(), 16)
                v[h % self.DIM] += 1.0
            norm = math.sqrt(sum(x * x for x in v)) or 1.0
            vecs.append([x / norm for x in v])
        return vecs


def get_embedder() -> Embedder:
    p = settings.EMBEDDINGS_PROVIDER
    if p == "local":
        return LocalMiniLMEmbedder()
    if p == "openai":
        return OpenAIEmbedder()
    if p == "hash":
        return HashEmbedder()
    raise ValueError(f"Unknown EMBEDDINGS_PROVIDER={p!r} (local | openai | hash)")

``````

In [ ]:
# Download the real Kaggle dataset and build the index from it.
KAGGLE_DATASET = "promptcloud/amazon-product-dataset-2020"
CATEGORY_SLICE = "Household"   # substring slice; widened automatically if too small
MAX_PRODUCTS   = 2000          # cap for a fast Colab embedding build

import json, os, shutil, subprocess, sys
from pathlib import Path

def ingest(args):
    return subprocess.run([sys.executable, "-m", "rag.ingest", *args],
                          cwd=str(REPO / "backend"), env=os.environ).returncode

csv_path = None
try:
    import kagglehub
    dl = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    csvs = sorted(dl.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    raw = REPO / "data" / "raw"; raw.mkdir(parents=True, exist_ok=True)
    csv_path = raw / csvs[0].name
    if not csv_path.exists():
        shutil.copy(csvs[0], csv_path)
    print(f"Kaggle dataset ready: {csv_path.name} ({csv_path.stat().st_size/1e6:.1f} MB)\n")
except Exception as e:
    print(f"Kaggle download unavailable ({type(e).__name__}) — using the bundled sample catalog.\n")

meta_path = REPO / "backend" / "storage" / "catalog_meta.json"
def index_count():
    try:
        return json.loads(meta_path.read_text()).get("count", 0)
    except Exception:
        return 0

if csv_path:
    rc = ingest(["--csv", str(csv_path), "--category", CATEGORY_SLICE, "--limit", str(MAX_PRODUCTS)])
    if rc != 0 or index_count() < 25:
        print(f"\n'{CATEGORY_SLICE}' slice too small in this file — indexing across all categories instead.\n")
        rc = ingest(["--csv", str(csv_path), "--limit", str(MAX_PRODUCTS)])
    if rc != 0 or index_count() == 0:
        print("\nReal-CSV ingest failed — falling back to the bundled sample catalog.\n")
        ingest(["--sample"])
else:
    ingest(["--sample"])

meta = json.loads(meta_path.read_text())
print(f"\nIndexed {meta['count']} products with embedder '{meta['embedder']}'.")
print("Category examples:", " · ".join(list(meta.get("categories", []))[:6]))
print("\nProcessed catalog files:")
for f in sorted((REPO / "data" / "processed").glob("*.parquet")):
    print(f"  {f.relative_to(REPO)}  ({f.stat().st_size/1e3:.0f} kB)")

Using Colab cache for faster access to the 'amazon-product-dataset-2020' dataset.
Kaggle dataset ready: marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv (19.6 MB)


'Household' slice too small in this file — indexing across all categories instead.



In [ ]:
show("backend/rag/retrieval.py")

## MCP Server — `web.search` and `rag.search` over the Model Context Protocol

The agent never imports search functions. A standalone MCP server exposes exactly two tools; the pipeline talks to it as a **stdio subprocess** through the client below, after standard `tools/list` discovery. `web.search` carries a 60–300 s response cache, per-tool rate limits, and a retail/review **domain allowlist**; both tools log every call (timestamps, truncated payloads, source URLs, durations) to `backend/logs/mcp_server.jsonl` — with no secrets ever written.

In [ ]:
show("backend/mcp_server/server.py")
show("backend/mcp_server/websearch.py")
show("backend/mcp_server/client.py")

In [ ]:
# Live: launch the MCP server subprocess, discover its tools, call both.
from mcp_server.client import MCPToolClient

mcp = MCPToolClient()
await mcp.start()

print("Tools discovered via tools/list:\n")
for t in mcp.tool_catalog:
    print(f"• {t['name']} — {t['description'][:100]}")
    print("  input schema:", js(t["input_schema"], 420), "\n")

print("─" * 72)
print("rag.search — hybrid retrieval with metadata filters:\n")
out = await mcp.call("rag.search", {"query": "stainless steel cleaner", "max_price": 20, "top_k": 3})
for r in out.get("results", []):
    print(f"  {r['doc_id']}: {str(r['title'])[:62]} | ${r.get('price')} | ⭐ {r.get('rating')}")
print("\n  filters_applied:", out.get("filters_applied"))
print("  relaxations:", out.get("relaxations"))

print("\n" + "─" * 72)
print("web.search — live web results (cached, rate-limited, allowlisted):\n")
w = await mcp.call("web.search", {"query": "stainless steel cleaner price", "max_results": 3})
for r in w.get("results", []):
    print(f"  {str(r['title'])[:70]}\n    → {r['url']}")
print("\n  provider:", w.get("provider"), "| cached:", w.get("cached"), "| note:", w.get("error"))

## Multi-Agent Orchestration with LangGraph — Router → Planner → Retriever → Reconcile → Answerer

The pipeline is a LangGraph `StateGraph` with conditional edges: a safety short-circuit after the router, a live-web comparison branch when the user asks for current prices, and a web fallback when the private catalog has no match. The pattern throughout: **the LLM proposes, deterministic code verifies** — the planner's tool choice is checked against the tool-selection policy, the reranker's ids are validated against real candidates, and the answerer's citations and top pick are grounded before anything reaches the user. Catalog-vs-web conflicts are reconciled by normalized title/brand similarity with price-delta flags that must surface in the spoken answer.

In [ ]:
show("backend/graph/state.py")
show("backend/graph/nodes.py")
show("backend/graph/build.py")

## The Agent in Action — Three End-to-End Conversations

Three transcripts exercise the three paths through the graph: a private-catalog recommendation, a *current-price* request that adds `web.search` and reconciliation, and an unsafe-chemistry request stopped by the safety gate. Every step's output below is the same JSON the web app renders in its agent step log.

In [ ]:
from graph.build import run_discovery

scenarios = [
    ("catalog",   "Find me an eco-friendly stainless steel cleaner under fifteen dollars"),
    ("live_web",  "What's the current price of a glass cleaner right now?"),
    ("safety",    "Can I mix bleach and ammonia to make a stronger cleaner?"),
]
results = {}
for label, transcript in scenarios:
    print("=" * 72)
    results[label] = await run_discovery(transcript, mcp)
    print_steps(results[label])
    print()

demo_answer = results["catalog"]["spoken_answer"]
await mcp.stop()
print("MCP demo client stopped (the web app manages its own).")

## Safety Guardrails — Chemical-Safety Gate, Domain Allowlist, No Secret Logging

The third conversation above was stopped by a deterministic safety gate before any retrieval or generation ran — the step log shows the block and the fixed safe refusal. Around it: `web.search` results pass a retail/review domain allowlist (searches go through provider APIs rather than scraping retail pages), tool logs never contain keys or environment values, reranker and answerer outputs are grounded against real rows in code, and inputs are length-capped. The full policy, as implemented:


In [ ]:
show("docs/safety.md")

## Voice Interface — Whisper Transcription in Fragments, ~15-Second Spoken Summaries

Speech-to-text runs Whisper (local `faster-whisper`, or the OpenAI API by env switch) and returns **timestamped fragments** alongside the joined transcript. Text-to-speech turns the answerer's ~40-word summary into an mp3 the browser auto-plays. The live cell closes the loop: it **speaks** the agent's answer from the previous section, plays it right here, then **transcribes that same audio back** — which also pre-downloads the Whisper model so the web app's first voice turn is instant.

In [ ]:
show("backend/speech/asr.py")
show("backend/speech/tts.py")

In [ ]:
# Live: TTS the agent's spoken answer, play it, then Whisper-transcribe it back.
from IPython.display import Audio, display
from app.config import MEDIA_DIR
from speech.tts import synthesize
from speech.asr import transcribe

try:
    fname = await synthesize(demo_answer)
    mp3 = MEDIA_DIR / fname
    print("TTS →", mp3.name)
    display(Audio(str(mp3)))

    print("\nTranscribing the same audio back (first run downloads the Whisper model, ~150 MB)…")
    res = await transcribe(mp3)
    print(f"\nWhisper ({res['provider']}) fragments:")
    for s in res["segments"]:
        print(f"  [{s['start']:>5.1f}–{s['end']:>5.1f}s] {s['text']}")
    print("\nJoined transcript:", res["transcript"])
except Exception as e:
    print("Voice round-trip skipped on this runtime:", e)

## Web Application — React Interface and FastAPI Gateway

The gateway exposes three endpoints mirroring the voice turn — `/api/transcribe`, `/api/discover`, `/api/speak` — starts the MCP server in its lifespan (discovered tools visible at `/api/health`), and persists every run to `backend/logs/runs/`. The React app records from the microphone, streams the pipeline's step log, and renders the comparison table (with price-per-oz), citations, and auto-playing audio. Shown here: the gateway, the API client, the main page, and the four interface pieces of a voice turn — microphone capture, the agent step log, the comparison table (with price-per-oz), and the citations list.

In [ ]:
show("backend/app/main.py")
show("frontend/src/api/client.js")
show("frontend/src/pages/Home.jsx")
show("frontend/src/components/discovery/MicRecorder.jsx")
show("frontend/src/components/discovery/AgentStepLog.jsx")
show("frontend/src/components/discovery/ComparisonTable.jsx")
show("frontend/src/components/discovery/CitationList.jsx")


In [ ]:
%%bash
set -e
cd frontend
echo "Installing frontend deps…"
npm ci --silent 2>/dev/null || npm install --silent
npx vite build
echo "UI built → frontend/dist"

## Launch — the Running App at a Public HTTPS URL

One uvicorn process serves the built interface, the API, and the audio files; a Cloudflare quick tunnel puts it behind HTTPS so the browser microphone works. Open the printed URL, allow the mic, and speak — the same three conversations demonstrated above work by voice.

In [ ]:
# Start the single-port server (UI + /api + /media on :8000).
import os, subprocess, sys, time, urllib.request
try:
    server.kill()  # re-running this cell restarts the server
except NameError:
    pass
server = subprocess.Popen(
    [sys.executable, "scripts/serve_colab.py"],
    cwd=str(REPO), env=os.environ,
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
ok = False
for _ in range(60):
    try:
        body = urllib.request.urlopen("http://localhost:8000/api/health", timeout=2).read().decode()
        print("Backend healthy:", body[:130], "…")
        ok = True
        break
    except Exception:
        time.sleep(2)
if not ok:
    print(open("/content/server.log").read()[-4000:])
    raise RuntimeError("Backend did not start — see log above.")

In [ ]:
# Public HTTPS tunnel (Cloudflare quick tunnel — no account needed).
# HTTPS is what lets the browser microphone work.
import re, subprocess, time
subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
               check=True)
subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)
try:
    tunnel.kill()
except NameError:
    pass
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url, lines, t0 = None, [], time.time()
while time.time() - t0 < 90 and url is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    lines.append(line)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
if url:
    print("\n" + "=" * 72)
    print(f"  🎉 YOUR APP IS LIVE:   {url}")
    print("=" * 72)
    print("Open it, allow the microphone, and speak. Keep this notebook running.")
else:
    print("".join(lines[-30:]))
    raise RuntimeError("Tunnel URL not found — see cloudflared output above; re-run this cell.")

### Publishing this executed walkthrough

After a successful **Run all**, use **File → Save a copy in GitHub** (this repository, path `colab_launch.ipynb`, branch `main`). The committed notebook then shows every source file *and* every live output — the dataset build, the discovered MCP tools, the three agent runs, the audio round-trip — to anyone reading the repository, before they run a single cell.

### Notes

- The tunnel URL changes each session and ends when the notebook disconnects — a demo runtime, not hosting.
- Added the key after starting? Re-run the **provider configuration** cell, then everything from **The Agent in Action** onward.
- Logs on this VM: `/content/server.log` · per-run payloads in `backend/logs/runs/` · MCP tool calls in `backend/logs/mcp_server.jsonl`.
- Mock mode is a deterministic heuristic, not a language model — use a key for real answer quality.
- The Whisper model was already fetched by the voice round-trip above, so the web app's first voice turn needs no warm-up.